<a href="https://colab.research.google.com/github/lnrdmnc/Quantum-Enhanced-Detection-Mechanisms-for-Mobile-Network-Security/blob/main/Quantum_Stumps_Tree_Boost_vs_VQS_VQE_QAOA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum Decision Stumps, Trees, and Boosting vs VQS/VQE/QAOA

This notebook implements **quantum decision stumps** (energy-based splits),
a **greedy tree** trained with **Gini gain**, and an **AdaBoost**-style ensemble.
We compare against **VQS**, **VQE**, and **QAOA** on:
- **PSD anomaly** (benign vs fake-SSB).
- **RACH detection** (match vs mismatch, reference-normalized).

In [1]:
import pennylane as qml
from pennylane import numpy as np
import numpy as base_np
import numpy.fft as fft
import matplotlib.pyplot as plt
print('PennyLane:', qml.__version__)

PennyLane: 0.42.3


In [2]:
%pip install pennylane

## RF helpers (PSD & RACH features)

In [3]:
def gen_ofdm_like(N=1024, subcarriers=(60,120,200), amps=(1.0,0.8,0.6), noise=0.05, fs=1.0):
    t = base_np.arange(N)/fs
    x = base_np.zeros(N, dtype=complex)
    for f,a in zip(subcarriers,amps):
        x += a * base_np.exp(1j*2*base_np.pi*f*t/N)
    x += noise * (base_np.random.randn(N) + 1j*base_np.random.randn(N))
    return x
def shift_sync(x, shift_bins=6):
    X = fft.fft(x); X = base_np.roll(X, shift_bins); return fft.ifft(X)
def add_tone_jammer(x, tone_bin=150, amp=3.0):
    N = len(x); n = base_np.arange(N)
    return x + amp*base_np.exp(1j*2*base_np.pi*tone_bin*n/N)
def psd_feature(x, n_bins=16):
    P = base_np.abs(fft.fft(x))**2
    k = len(P)//n_bins
    P = P[:k*n_bins].reshape(n_bins, k).mean(axis=1)
    P = P/(P.max()+1e-12)
    return P.astype(float)
def zadoff_chu(u,Nzc):
    n = base_np.arange(Nzc)
    return base_np.exp(-1j*base_np.pi*u*n*(n+1)/Nzc)
def rach_burst(zc, delay=0, snr_db=12.0):
    N=len(zc); x=base_np.zeros(N, dtype=complex); x[delay:]=zc[:N-delay]
    snr_lin = 10**(snr_db/10)
    sig_power = base_np.mean(base_np.abs(x)**2)+1e-12
    noise_power = sig_power/snr_lin
    noise = base_np.sqrt(noise_power/2)*(base_np.random.randn(N)+1j*base_np.random.randn(N))
    return x + noise
def corr_feature_refnorm(rx, ref, n_bins=16, ref_scale=None):
    r = base_np.correlate(rx, ref, mode='same'); c = base_np.abs(r)
    if ref_scale is None:
        rref = base_np.correlate(ref, ref, mode='same'); ref_scale = float(base_np.max(base_np.abs(rref)))
    c = c/(ref_scale+1e-12)
    k = len(c)//n_bins
    return c[:k*n_bins].reshape(n_bins,k).mean(axis=1).astype(float)

## Ising Hamiltonians and circuits (VQE & QAOA)

In [4]:
def build_ising_H_from_stats(mean_vec, cov_mat, wires, kappa=1.0):
    H = 0
    mu = mean_vec - mean_vec.mean()
    for wi, a in zip(wires, mu):
        H = H + float(a) * qml.PauliZ(wi)
    for i in range(len(wires)-1):
        J = kappa * float(cov_mat[i, i+1])
        if abs(J) > 1e-12:
            H = H + J * (qml.PauliZ(wires[i]) @ qml.PauliZ(wires[i+1]))
    return H

def angle_embed(x, wires):
    qml.AngleEmbedding(x, wires=wires, rotation='Y')

def ansatz_vqe(theta, L, wires):
    n=len(wires); idx=0
    for _ in range(L):
        for w in wires: qml.RZ(theta[idx], wires=w); idx+=1
        for w in wires: qml.RX(theta[idx], wires=w); idx+=1
        for i in range(n): qml.CNOT(wires=[wires[i], wires[(i+1)%n]])
        for i in range(n): qml.IsingZZ(0.25, wires=[wires[i], wires[(i+1)%n]])

def qaoa_block(gammas, betas, H, wires):
    mixer_h = qml.qaoa.x_mixer(wires)
    for g,b in zip(gammas, betas):
        qml.qaoa.cost_layer(g, H)
        qml.qaoa.mixer_layer(b, mixer_h)

def scalar(x):
    try: return float(x)
    except Exception: return float(np.array(x))

## Datasets (train/val)

In [5]:
def build_psd_dataset(B=16, N=1024, n_train=80, n_val=20):
    def sample(kind, n):
        S=[]
        for _ in range(n):
            x=gen_ofdm_like(N=N)
            if kind=='fake': x=shift_sync(x,6)
            S.append(psd_feature(x, n_bins=B))
        return base_np.stack(S,axis=0)
    S_b_tr=sample('benign', n_train); S_b_val=sample('benign', n_val)
    S_f_tr=sample('fake',   n_train); S_f_val=sample('fake',   n_val)
    mu_b, mu_f = S_b_tr.mean(0), S_f_tr.mean(0)
    Cb, Cf = base_np.cov(S_b_tr.T), base_np.cov(S_f_tr.T)
    return (S_b_tr,S_f_tr,S_b_val,S_f_val, mu_b,mu_f,Cb,Cf)

def build_rach_dataset(B=16, Nzc=839, u_root=167, n_pos=28, n_neg=40):
    zc=zadoff_chu(u_root, Nzc)
    rref=base_np.correlate(zc,zc,mode='same'); ref_scale=float(base_np.max(base_np.abs(rref)))
    R_pos=base_np.stack([corr_feature_refnorm(rach_burst(zc, 12+d, 12.0), zc, n_bins=B, ref_scale=ref_scale) for d in range(-3,4)],axis=0)
    R_neg=base_np.stack([corr_feature_refnorm(rach_burst(zc, base_np.random.randint(40,120), 8.0), zc, n_bins=B, ref_scale=ref_scale) for _ in range(n_neg)],axis=0)
    Rp_tr, Rp_val = R_pos[:len(R_pos)//2], R_pos[len(R_pos)//2:]
    Rn_tr, Rn_val = R_neg[:len(R_neg)//2], R_neg[len(R_neg)//2:]
    mu_p, mu_n = Rp_tr.mean(0), Rn_tr.mean(0)
    Cp, Cn = base_np.cov(Rp_tr.T), base_np.cov(Rn_tr.T)
    return (zc, ref_scale, Rp_tr, Rn_tr, Rp_val, Rn_val, mu_p,mu_n,Cp,Cn)

## Baselines: VQE/QAOA ΔE for PSD and RACH

In [ ]:
# PSD baselines
base_np.random.seed(1)
B=16; wires=list(range(B))
S_b_tr,S_f_tr,S_b_val,S_f_val, mu_b,mu_f, Cb,Cf = build_psd_dataset(B=B)
H_b = build_ising_H_from_stats(mu_b,Cb,wires); H_f = build_ising_H_from_stats(mu_f,Cf,wires)

dev_vqe_psd = qml.device('default.qubit', wires=B)
@qml.qnode(dev_vqe_psd)
def energy_vqe(th, x, H): angle_embed(x,wires); ansatz_vqe(th,2,wires); return qml.expval(H)
def batch_E_vqe(th, X, H): return np.mean(np.stack([energy_vqe(th, x, H) for x in X]))
def loss_vqe(th, Xb, Xf, margin=0.15):
    Eb=batch_E_vqe(th,Xb,H_b); Ef=batch_E_vqe(th,Xf,H_f)
    return Eb + qml.math.maximum(0.0, margin - (Ef - Eb))
np.random.seed(2)
theta_psd = np.random.uniform(-0.1,0.1, size=2*B*2)
opt = qml.AdamOptimizer(stepsize=0.05)
for s in range(40):
    i=(s*8)%64; Xb=S_b_tr[i:i+8]; Xf=S_f_tr[i:i+8]
    theta_psd,_ = opt.step_and_cost(lambda th: loss_vqe(th,Xb,Xf), theta_psd)

dev_q_psd = qml.device('default.qubit', wires=B)
p=2
@qml.qnode(dev_q_psd)
def energy_qaoa(pr, x, H):
    gam=pr[:p]; bet=pr[p:2*p]
    angle_embed(x,wires); qaoa_block(gam,bet,H,wires); return qml.expval(H)
def batch_E_qaoa(pr, X, H): return np.mean(np.stack([energy_qaoa(pr, x, H) for x in X]))
def loss_qaoa(pr, Xb, Xf, margin=0.15):
    Eb=batch_E_qaoa(pr,Xb,H_b); Ef=batch_E_qaoa(pr,Xf,H_f)
    return Eb + qml.math.maximum(0.0, margin - (Ef - Eb))
np.random.seed(3)
params_psd = np.random.uniform(-0.1,0.1, size=2*p)
optq = qml.AdamOptimizer(stepsize=0.05)
for s in range(40):
    i=(s*8)%64; Xb=S_b_tr[i:i+8]; Xf=S_f_tr[i:i+8]
    params_psd,_ = optq.step_and_cost(lambda pr: loss_qaoa(pr,Xb,Xf), params_psd)

def deltaE_vqe_psd(x): return float(energy_vqe(theta_psd,x,H_f) - energy_vqe(theta_psd,x,H_b))
def deltaE_qaoa_psd(x): return float(energy_qaoa(params_psd,x,H_f) - energy_qaoa(params_psd,x,H_b))

# RACH baselines
zc, ref_scale, Rp_tr, Rn_tr, Rp_val, Rn_val, mu_p,mu_n,Cp,Cn = build_rach_dataset(B=B)
H_pos = build_ising_H_from_stats(mu_p,Cp,wires); H_neg = build_ising_H_from_stats(mu_n,Cn,wires)
dev_vqe_r = qml.device('default.qubit', wires=B)
@qml.qnode(dev_vqe_r)
def energy_vqe_r(th, x, H): angle_embed(x,wires); ansatz_vqe(th,2,wires); return qml.expval(H)
def loss_vqe_r(th, Rp, Rn, margin=0.08):
    Ep=np.mean(np.stack([energy_vqe_r(th,x,H_pos) for x in Rp]))
    En=np.mean(np.stack([energy_vqe_r(th,x,H_neg) for x in Rn]))
    return Ep + qml.math.maximum(0.0, margin - (En - Ep))
np.random.seed(4)
theta_rach = np.random.uniform(-0.1,0.1, size=2*B*2)
opt2 = qml.AdamOptimizer(stepsize=0.05)
for s in range(40):
    i=(s*6)%len(Rp_tr); RpB=Rp_tr[max(0,i-3):i+4]; RnB=Rn_tr[i:i+7]
    theta_rach,_ = opt2.step_and_cost(lambda th: loss_vqe_r(th,RpB,RnB), theta_rach)

dev_q_r = qml.device('default.qubit', wires=B)
pR=2
@qml.qnode(dev_q_r)
def energy_qaoa_r(pr, x, H):
    gam=pr[:pR]; bet=pr[pR:2*pR]
    angle_embed(x,wires); qaoa_block(gam,bet,H,wires); return qml.expval(H)
def loss_qaoa_r(pr, Rp, Rn, margin=0.08):
    Ep=np.mean(np.stack([energy_qaoa_r(pr,x,H_pos) for x in Rp]))
    En=np.mean(np.stack([energy_qaoa_r(pr,x,H_neg) for x in Rn]))
    return Ep + qml.math.maximum(0.0, margin - (En - Ep))
np.random.seed(5)
params_rach = np.random.uniform(-0.1,0.1, size=2*pR)
optR = qml.AdamOptimizer(stepsize=0.05)
for s in range(40):
    i=(s*6)%len(Rp_tr); RpB=Rp_tr[max(0,i-3):i+4]; RnB=Rn_tr[i:i+7]
    params_rach,_ = optR.step_and_cost(lambda pr: loss_qaoa_r(pr,RpB,RnB), params_rach)

def deltaE_vqe_rach(x): return float(energy_vqe_r(theta_rach,x,H_neg) - energy_vqe_r(theta_rach,x,H_pos))
def deltaE_qaoa_rach(x): return float(energy_qaoa_r(params_rach,x,H_neg) - energy_qaoa_r(params_rach,x,H_pos))

## Quantum Decision Stump, Greedy Tree, and AdaBoost

In [ ]:
def gini_impurity(y, w=None):
    y = base_np.asarray(y)
    if w is None: w = base_np.ones_like(y, dtype=float)
    w = base_np.asarray(w, dtype=float)
    W = w.sum() + 1e-12
    p1 = (w[y==1].sum())/W
    return 2*p1*(1-p1)

def best_threshold(scores, y, w=None):
    if w is None: w = base_np.ones_like(scores, dtype=float)
    idx = base_np.argsort(scores)
    s_sorted = scores[idx]; y_sorted = y[idx]; w_sorted = w[idx]
    W = w_sorted.sum();
    w1 = (w_sorted * (y_sorted==1)).astype(float)
    prefix_w  = base_np.cumsum(w_sorted)
    prefix_w1 = base_np.cumsum(w1)
    best_gain=-1.0; best_tau=None
    G_parent = gini_impurity(y, w)
    for i in range(1, len(scores)):
        if s_sorted[i]==s_sorted[i-1]:
            continue
        WL = prefix_w[i-1]; WR = W-WL
        if WL<1e-9 or WR<1e-9:
            continue
        w1L = prefix_w1[i-1]; w1R = (w1.sum()-w1L)
        pL = w1L/(WL+1e-12); pR = w1R/(WR+1e-12)
        G_L = 2*pL*(1-pL); G_R = 2*pR*(1-pR)
        G = G_parent - (WL/W)*G_L - (WR/W)*G_R
        if G>best_gain:
            best_gain=G
            best_tau=0.5*(s_sorted[i]+s_sorted[i-1])
    if best_tau is None:
        best_tau = float(scores.mean())
        best_gain = 0.0
    return best_tau, float(best_gain)

class QuantumStump:
    def __init__(self, H_pos, H_neg, wires, backend='qaoa', p=2, L=2, steps=30, margin=0.1, lr=0.05):
        self.H_pos=H_pos; self.H_neg=H_neg; self.wires=wires
        self.backend=backend; self.p=p; self.L=L; self.steps=steps; self.margin=margin; self.lr=lr
        self.tau=0.0; self.params=None
        if backend=='qaoa':
            self.dev = qml.device('default.qubit', wires=len(wires))
            @qml.qnode(self.dev)
            def Ecost(params, x, H):
                gam=params[:p]; bet=params[p:2*p]
                angle_embed(x,wires); qaoa_block(gam,bet,H,wires); return qml.expval(H)
            self._E = Ecost
            np.random.seed(123)
            self.params = np.random.uniform(-0.1,0.1, size=2*p)
            self.opt = qml.AdamOptimizer(stepsize=lr)
        else:
            self.dev = qml.device('default.qubit', wires=len(wires))
            @qml.qnode(self.dev)
            def Ecost(theta, x, H):
                angle_embed(x,wires); ansatz_vqe(theta,L,wires); return qml.expval(H)
            self._E = Ecost
            np.random.seed(321)
            self.params = np.random.uniform(-0.1,0.1, size=2*len(wires)*L)
            self.opt = qml.AdamOptimizer(stepsize=lr)
    def score(self, X):
        return base_np.array([float(self._E(self.params, x, self.H_neg) - self._E(self.params, x, self.H_pos)) for x in X])
    def fit(self, X, y, sample_weight=None):
        y = base_np.asarray(y).astype(int)
        if sample_weight is None:
            sample_weight = base_np.ones_like(y, dtype=float)
        def loss_fn(params):
            s = [self._E(params, x, self.H_neg) - self._E(params, x, self.H_pos) for x in X]
            s = np.stack(s)
            yy = 2*y-1
            w = sample_weight
            margins = yy * s
            hinge = qml.math.maximum(0.0, self.margin - margins)
            return (w * hinge).mean()
        for _ in range(self.steps):
            self.params, _ = self.opt.step_and_cost(loss_fn, self.params)
        scores = self.score(X)
        self.tau, gain = best_threshold(scores, y, w=sample_weight)
        self.gain_ = gain
        left = y[scores<=self.tau]
        right = y[scores>self.tau]
        self.left_label = int(base_np.round(left.mean())) if len(left)>0 else 0
        self.right_label= int(base_np.round(right.mean())) if len(right)>0 else 1
        return self
    def predict(self, X):
        s = self.score(X)
        return base_np.where(s<=self.tau, self.left_label, self.right_label)

class GreedyQuantumTree:
    def __init__(self, stump_candidates, max_depth=2, min_gain=1e-3, min_samples_leaf=5):
        self.stump_candidates = stump_candidates
        self.max_depth = max_depth; self.min_gain=min_gain; self.min_samples_leaf=min_samples_leaf
        self.root=None
    def _build(self, X, y, w, depth):
        best=None; best_model=None
        for make_stump in self.stump_candidates:
            stump = make_stump()
            stump.fit(X, y, sample_weight=w)
            if best is None or stump.gain_ > best:
                best = stump.gain_; best_model = stump
        node = {'stump': best_model, 'left': None, 'right': None}
        if depth>=self.max_depth or best<self.min_gain:
            node['leaf'] = int(base_np.round(base_np.average(y, weights=w)))
            return node
        s = best_model.score(X)
        L = s<=best_model.tau; R = ~L
        if L.sum()<self.min_samples_leaf or R.sum()<self.min_samples_leaf:
            node['leaf'] = int(base_np.round(base_np.average(y, weights=w)))
            return node
        node['left']  = self._build(X[L], y[L], w[L], depth+1)
        node['right'] = self._build(X[R], y[R], w[R], depth+1)
        return node
    def fit(self, X, y, sample_weight=None):
        if sample_weight is None: sample_weight = base_np.ones_like(y, dtype=float)
        self.root = self._build(X, y, sample_weight, 0)
        return self
    def _predict_node(self, node, x):
        if 'leaf' in node and node['leaf'] is not None:
            return node['leaf']
        stump = node['stump']
        s = stump.score([x])[0]
        if s<=stump.tau: return self._predict_node(node['left'], x)
        else: return self._predict_node(node['right'], x)
    def predict(self, X):
        return base_np.array([self._predict_node(self.root, x) for x in X])

class QuantumAdaBoost:
    def __init__(self, stump_factory, rounds=5):
        self.stump_factory = stump_factory; self.rounds=rounds
        self.models=[]; self.alphas=[]
    def fit(self, X, y):
        n=len(y); w = base_np.ones(n)/n
        for _ in range(self.rounds):
            stump = self.stump_factory()
            stump.fit(X, y, sample_weight=w)
            y_pred = stump.predict(X)
            err = (w*(y_pred!=y)).sum() + 1e-12
            alpha = 0.5*base_np.log((1-err)/err)
            self.models.append(stump); self.alphas.append(alpha)
            w *= base_np.exp(-alpha*(2*y-1)*(2*y_pred-1))
            w /= w.sum()
        return self
    def decision_function(self, X):
        f = base_np.zeros(len(X))
        for stump,alpha in zip(self.models, self.alphas):
            pred = stump.predict(X)
            f += alpha*(2*pred-1)
        return f
    def predict(self, X):
        return (self.decision_function(X)>0).astype(int)

## Train & compare on PSD and RACH

In [ ]:
def auroc(scores_pos, scores_neg):
    s = base_np.concatenate([scores_pos, scores_neg])
    y = base_np.concatenate([
        base_np.ones_like(scores_pos),
        base_np.zeros_like(scores_neg)
    ])

    order = base_np.argsort(s)
    s, y = s[order], y[order]
    uniq = base_np.unique(s)

    tpr, fpr = [], []
    P = y.sum()
    N = len(y) - P

    for t in uniq:
        pred = (s >= t).astype(int)
        TP = (pred * y).sum()
        FP = (pred * (1 - y)).sum()
        tpr.append(TP / (P + 1e-12))
        fpr.append(FP / (N + 1e-12))

    order = base_np.argsort(fpr)
    fpr = base_np.array(fpr)[order]
    tpr = base_np.array(tpr)[order]

    # correzione: trapz è deprecato → trapezoid
    auc = base_np.trapezoid(tpr, fpr)

    return float(auc), fpr, tpr


# =====================
# PSD task
# =====================
y_tr = base_np.concatenate([
    base_np.zeros(len(S_b_tr), dtype=int),
    base_np.ones(len(S_f_tr), dtype=int)
])
X_tr = base_np.concatenate([S_b_tr, S_f_tr], axis=0)

y_val = base_np.concatenate([
    base_np.zeros(len(S_b_val), dtype=int),
    base_np.ones(len(S_f_val), dtype=int)
])
X_val = base_np.concatenate([S_b_val, S_f_val], axis=0)

dE_vqe_psd_val = base_np.array([deltaE_vqe_psd(x) for x in X_val])
dE_qaoa_psd_val = base_np.array([deltaE_qaoa_psd(x) for x in X_val])

auc_vqe_psd, fpr1, tpr1 = auroc(
    dE_vqe_psd_val[y_val == 1],
    dE_vqe_psd_val[y_val == 0]
)
auc_qaoa_psd, fpr2, tpr2 = auroc(
    dE_qaoa_psd_val[y_val == 1],
    dE_qaoa_psd_val[y_val == 0]
)


def stump_qaoa_factory():
    return QuantumStump(
        H_pos=H_b,
        H_neg=H_f,
        wires=wires,
        backend="qaoa",
        p=2,
        steps=25,
        margin=0.15,
    )


tree = GreedyQuantumTree([stump_qaoa_factory], max_depth=2)
tree.fit(X_tr, y_tr)
pred_tree = tree.predict(X_val)
auc_tree, fprT, tprT = auroc(pred_tree[y_val == 1], pred_tree[y_val == 0])

boost = QuantumAdaBoost(stump_qaoa_factory, rounds=5).fit(X_tr, y_tr)
score_boost = boost.decision_function(X_val)
auc_boost, fprB, tprB = auroc(score_boost[y_val == 1], score_boost[y_val == 0])

print({
    "PSD AUROC": {
        "VQE": round(auc_vqe_psd, 3),
        "QAOA": round(auc_qaoa_psd, 3),
        "Tree": round(auc_tree, 3),
        "Boost": round(auc_boost, 3),
    }
})

plt.figure(figsize=(6, 4))
plt.plot(fpr1, tpr1, label=f"VQE {auc_vqe_psd:.2f}")
plt.plot(fpr2, tpr2, label=f"QAOA {auc_qaoa_psd:.2f}")
plt.plot(fprT, tprT, label=f"Tree {auc_tree:.2f}")
plt.plot(fprB, tprB, label=f"Boost {auc_boost:.2f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("PSD ROC")
plt.legend()
plt.show()


# =====================
# RACH task
# =====================
y_tr_r = base_np.concatenate([
    base_np.zeros(len(Rp_tr), dtype=int),
    base_np.ones(len(Rn_tr), dtype=int)
])
X_tr_r = base_np.concatenate([Rp_tr, Rn_tr], axis=0)

y_val_r = base_np.concatenate([
    base_np.zeros(len(Rp_val), dtype=int),
    base_np.ones(len(Rn_val), dtype=int)
])
X_val_r = base_np.concatenate([Rp_val, Rn_val], axis=0)

dE_vqe_r_val = base_np.array([deltaE_vqe_rach(x) for x in X_val_r])
dE_qaoa_r_val = base_np.array([deltaE_qaoa_rach(x) for x in X_val_r])

auc_vqe_r, fpr1r, tpr1r = auroc(
    dE_vqe_r_val[y_val_r == 1],
    dE_vqe_r_val[y_val_r == 0]
)
auc_qaoa_r, fpr2r, tpr2r = auroc(
    dE_qaoa_r_val[y_val_r == 1],
    dE_qaoa_r_val[y_val_r == 0]
)


def stump_qaoa_factory_r():
    return QuantumStump(
        H_pos=H_pos,
        H_neg=H_neg,
        wires=wires,
        backend="qaoa",
        p=2,
        steps=25,
        margin=0.08,
    )


tree_r = GreedyQuantumTree([stump_qaoa_factory_r], max_depth=2)
tree_r.fit(X_tr_r, y_tr_r)
pred_tree_r = tree_r.predict(X_val_r)
auc_tree_r, fprTr, tprTr = auroc(pred_tree_r[y_val_r == 1], pred_tree_r[y_val_r == 0])

boost_r = QuantumAdaBoost(stump_qaoa_factory_r, rounds=5).fit(X_tr_r, y_tr_r)
score_boost_r = boost_r.decision_function(X_val_r)
auc_boost_r, fprBr, tprBr = auroc(score_boost_r[y_val_r == 1], score_boost_r[y_val_r == 0])

print({
    "RACH AUROC": {
        "VQE": round(auc_vqe_r, 3),
        "QAOA": round(auc_qaoa_r, 3),
        "Tree": round(auc_tree_r, 3),
        "Boost": round(auc_boost_r, 3),
    }
})

plt.figure(figsize=(6, 4))
plt.plot(fpr1r, tpr1r, label=f"VQE {auc_vqe_r:.2f}")
plt.plot(fpr2r, tpr2r, label=f"QAOA {auc_qaoa_r:.2f}")
plt.plot(fprTr, tprTr, label=f"Tree {auc_tree_r:.2f}")
plt.plot(fprBr, tprBr, label=f"Boost {auc_boost_r:.2f}")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("RACH ROC")
plt.legend()
plt.show()
